# T3. 쇼핑몰 CDC → AI 상담봇

교재 1권의 쇼핑몰 주문 1,000건을 MySQL 에 넣고 Debezium 으로 변경을 실시간 캡처해 MinIO 위 Iceberg 테이블에 반영합니다. 가상 쇼핑몰의 배송 · 환불 규정 문서를 Qdrant 에 넣어, 주문 조회(SQL)와 규정 검색(RAG)을 함께 쓰는 상담봇을 Streamlit 으로 만듭니다.

- 단계: 팀 프로젝트 · 정형 + 비정형 · 예상 3일(팀 4~5인)
- 실행 환경: 내 컴퓨터의 Docker + Jupyter(VS Code 노트북도 됩니다)
- 데이터: 교재 1권 12장 · 16장 쇼핑몰 주문 데이터 orders_clean.parquet(시드를 고정해 만든 연습용 합성 데이터 1,000건)
- 데이터: 배송 상태 갱신 · 가상 쇼핑몰 FAQ · 환불 규정 문서는 이 프로젝트에서 연습용으로 만든 것

**빈칸 노트북** — 코드 속 `___①___` 자리 18곳을 채웁니다. 줄 끝 주석이 힌트입니다.
막히면 [따라하기 노트북](https://colab.research.google.com/github/aebonlee/materials/blob/main/data-eng/projects/T3/T3_guide.ipynb)의 같은 단계를 보세요. 정답은 맨 아래에 있습니다.

쇼핑몰의 주문 DB(MySQL)는 주문이 들어오고 배송 상태가 바뀔 때마다 고쳐집니다. 분석 쪽에서 그 변화를 따라가려고 매일 밤 표 전체를 복사하면 늦고 무겁습니다.
**CDC(변경 데이터 캡처)** 는 DB 의 변경 기록(binlog)을 읽어 「무엇이 어떻게 바뀌었는지」만 이벤트로 흘려 줍니다(6주차 6-3 · 6-4).
이 프로젝트는 그 이벤트를 레이크하우스(Iceberg)에 반영하고, 그 위에 「내 주문 언제 와?」 「환불 규정은?」에 답하는 상담봇을 올립니다.

```text
MySQL shop.orders ─(binlog)─▶ Debezium(Kafka Connect) ─▶ Kafka shop.shop.orders ─▶ 반영기 ─▶ Iceberg on MinIO
                                                                                            │ 주문 조회(SQL)
가상 쇼핑몰 FAQ · 환불 규정 ─▶ 청킹 · 임베딩 ─▶ Qdrant ─────────── 규정 검색(RAG) ──▶ 상담봇(Streamlit)
```

팀 프로젝트라 역할을 넷으로 나눕니다. 단계 제목 옆의 역할이 그 단계의 담당입니다. 처음에는 한 사람이 노트북 하나로 끝까지 돌려 보고, 그다음 역할별로 나눠 키우세요.

| 역할 | 맡는 단계 |
|---|---|
| 수집 · 스트리밍 | 3 MySQL 적재 · 4 Debezium 커넥터 |
| 레이크 · Spark | 2 인프라 · 5 Iceberg 반영 · 6 변경 따라가기 |
| RAG · 서비스 | 7 규정 문서 · 8 상담봇 · 9 Streamlit |
| 오케스트레이션 | 10 Airflow DAG · 11 정리 |

다 끝내면 다음을 말할 수 있습니다.

- DB 의 `UPDATE` 한 줄이 Kafka 에 어떤 모양(op · before · after)의 이벤트로 나오는지
- 이벤트를 받아 레이크하우스 표를 원본 DB 와 **같은 상태**로 맞추는 방법(upsert · delete)과, 맞았는지 확인하는 방법
- 상담봇이 질문마다 SQL 과 RAG 중 무엇을 쓰는지, 그리고 왜 본인 주문만 보여 줘야 하는지

> **자주 겪는 오류** — 이 노트북은 **내 컴퓨터의 Docker** 에서 돕니다(Colab 에서는 MySQL · Kafka 를 띄울 수 없습니다). Docker Desktop 을 켜 두고, 메모리는 6GB 이상 주세요.
> 데이터 출처를 밝혀 둡니다 — 주문 1,000건은 교재 1권 12장에서 시드를 고정해 만든 **연습용 합성 데이터**(16장 정제본)입니다. 배송 상태와 그 변화, 쇼핑몰 규정 문서는 이 프로젝트를 위해 만든 **가상의 것**이며 실제 쇼핑몰과 관계가 없습니다.

## 1단계 · 준비 — 라이브러리와 주문 데이터

필요한 라이브러리는 2주차(pymysql) · 6주차(confluent-kafka) · 18장 Iceberg(이번에는 파이썬판 PyIceberg) · 7주차(qdrant-client · sentence-transformers) · 8주차(streamlit)에서 쓴 것입니다. 없는 것만 설치합니다.

In [ ]:
import sys, subprocess, importlib.util
def need(mod, pkg=None):
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg or mod], check=True)
need("pymysql")
need("cryptography")                              # MySQL 8 이후 기본 로그인 방식(caching_sha2)에 필요
need("confluent_kafka", "confluent-kafka")
need("pyiceberg", "pyiceberg[sql-sqlite,pyarrow]")
need("qdrant_client", "qdrant-client")
need("sentence_transformers", "sentence-transformers")
need("streamlit")

import os, io, re, json, time, socket, shutil, urllib.request
from collections import Counter
from datetime import date, datetime, timedelta, timezone
import pandas as pd
import pyarrow as pa
import duckdb

os.environ["TOKENIZERS_PARALLELISM"] = "false"
DATA = "https://raw.githubusercontent.com/aebonlee/materials/main/data-eng/projects/data/"

def fetch(name):
    """프로젝트 데이터를 ./data 폴더에 한 번만 받아 둡니다."""
    os.makedirs("data", exist_ok=True)
    dst = os.path.join("data", name)
    if not os.path.exists(dst):
        local = os.environ.get("DE_DATA_LOCAL")
        if local:
            shutil.copy(os.path.join(local, name), dst)
        else:
            urllib.request.urlretrieve(DATA + name, dst)
    return dst

orders = pd.read_parquet(fetch("shop_orders_clean.parquet"))
print(orders.shape, "· 합계", f"{int(orders['amount'].sum()):,}원")
orders.head(3)

16장 끝에서 만든 정제본 그대로입니다 — 중복 15건을 빼고 가격 결측 30건을 중앙값으로 채운 1,000건입니다(`price_imputed` 가 채운 표시).

> **확인 포인트** — `(1000, 8) · 합계 291,021,500원` — 16장 품질 리포트의 「after 1,000행」과 같습니다.

In [ ]:
assert orders.shape == (1000, 8) and int(orders["amount"].sum()) == 291021500
assert shutil.which("docker"), "Docker 가 없습니다 — Docker Desktop 을 설치하고 켜 주세요"

## 2단계 · 인프라 띄우기 — MySQL · Kafka · Debezium · MinIO · Qdrant (레이크 · Spark)

6주차 6-4 의 구성(kafka · mysql · connect)에 MinIO 와 Qdrant 를 더합니다. Kafka 설정은 6-2 의 리스너 3개 그대로입니다.
MySQL 은 내 노트북 쪽 포트를 **3307** 로 엽니다 — 2주차의 `de-mysql` 이 3306 을 쓰고 있어도 부딪히지 않게 하려는 것입니다(6주차 6-4 의 경고). 컨테이너끼리는 여전히 `mysql:3306` 입니다.

In [ ]:
%%writefile docker-compose.yml
services:
  kafka:
    image: apache/kafka:3.7.0
    ports:
      - "9092:9092"
    environment:
      KAFKA_NODE_ID: 1
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@kafka:29093
      KAFKA_LISTENERS: PLAINTEXT://:29092,CONTROLLER://:29093,EXTERNAL://:9092
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://kafka:29092,EXTERNAL://localhost:9092
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: PLAINTEXT:PLAINTEXT,CONTROLLER:PLAINTEXT,EXTERNAL:PLAINTEXT
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_INTER_BROKER_LISTENER_NAME: PLAINTEXT
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 1
  mysql:
    image: mysql:8.0
    ports:
      - "___①___"   # ① 내 노트북 3307 → 컨테이너 3306
    environment:
      MYSQL_ROOT_PASSWORD: rootpw
      MYSQL_DATABASE: shop
    volumes:
      - mysql-data:/var/lib/mysql
  connect:
    image: debezium/connect:2.7.3.Final
    depends_on:
      - kafka
      - mysql
    ports:
      - "8083:8083"
    environment:
      BOOTSTRAP_SERVERS: ___②___   # ② 컨테이너 안에서 붙는 Kafka 주소
      GROUP_ID: 1
      CONFIG_STORAGE_TOPIC: connect_configs
      OFFSET_STORAGE_TOPIC: connect_offsets
      STATUS_STORAGE_TOPIC: connect_statuses
  minio:
    image: pgsty/minio:RELEASE.2026-08-04T00-00-00Z
    command: server /data --console-address ":9001"
    ports:
      - "9000:9000"
      - "9001:9001"
    environment:
      MINIO_ROOT_USER: minioadmin
      MINIO_ROOT_PASSWORD: minioadmin
    volumes:
      - minio-data:/data
  qdrant:
    image: qdrant/qdrant:v1.19.1
    ports:
      - "6333:6333"
  airflow:
    image: apache/airflow:2.9.3
    profiles: ["airflow"]
    volumes:
      - ./dags:/opt/airflow/dags
      - ./:/opt/t3
    environment:
      AIRFLOW__CORE__LOAD_EXAMPLES: "False"
volumes:
  mysql-data:
  minio-data:

> **자주 겪는 오류** — MySQL 은 6주차 6-4 와 같은 `mysql:8.0` 입니다. 2주차의 `mysql:9.7` 로 바꾸면 Debezium 2.7.3 이 스냅숏을 시작하지 못합니다 — MySQL 8.4 부터 없어진 `SHOW MASTER STATUS` 를 부르다 문법 오류가 나기 때문입니다(직접 확인).
> 더 고약한 점은 커넥터 상태가 계속 `RUNNING` 으로 보인다는 것입니다(재시도 가능한 오류라 스스로 다시 시작합니다). 토픽이 안 생기면 `docker compose -p t3 logs connect` 에서 `MASTER STATUS` 를 찾아보세요.

In [ ]:
!docker compose --progress quiet -p t3 up -d
!docker compose -p t3 ps --services --status running

MySQL 은 처음 뜰 때 DB 를 만드느라 한 번 재시작하고, Connect 는 Kafka 에 설정 토픽을 만든 뒤에야 REST 를 엽니다. 둘 다 실제로 응답할 때까지 기다립니다.

In [ ]:
import pymysql, requests

def wait(fn, sec=180, what=""):
    t0, last = time.time(), None
    while time.time() - t0 < sec:
        try:
            return fn()
        except Exception as e:
            last = e
            time.sleep(2)
    raise TimeoutError(f"{what} {sec}초 안에 준비되지 않음: {last}")

def db(user="root", pw="rootpw"):
    return pymysql.connect(host="127.0.0.1", port=3307, user=user, password=pw, database="shop", autocommit=True)

def mysql_version():
    with db() as c:
        return c.get_server_info()

ver = wait(mysql_version, what="MySQL")
CONNECT = "http://localhost:8083"
plugins = wait(lambda: requests.get(f"{CONNECT}/connector-plugins", timeout=5).json(), what="Connect")
print("MySQL", ver)
print([p["class"] for p in plugins if "mysql" in p["class"].lower()])

> **확인 포인트** — `MySQL 8.0.…` 과 `['io.debezium.connector.mysql.MySqlConnector']` — 6주차 6-4 1단계의 플러그인 목록과 같습니다.

In [ ]:
assert ver.startswith("8.0") and "io.debezium.connector.mysql.MySqlConnector" in [p["class"] for p in plugins]

## 3단계 · MySQL 적재 — 주문 표와 CDC 계정 (수집 · 스트리밍)

원본 주문에는 배송 상태가 없습니다. 상담봇이 「언제 와?」에 답하려면 상태가 있어야 하므로 `status`(처음엔 모두 `결제완료`)와 `updated_at` 열을 더합니다 — **연습용으로 더한 열**입니다.
금액은 원 단위 정수라 `INT` 로 둡니다. `DECIMAL` 로 두면 Debezium 이 기본 설정에서 숫자를 바이트(base64)로 바꿔 보내 읽기가 번거롭습니다.

In [ ]:
DDL = """
CREATE TABLE IF NOT EXISTS orders (
  order_id     INT PRIMARY KEY,
  customer_id  INT NOT NULL,
  category     VARCHAR(10) NOT NULL,
  price        INT NOT NULL,
  quantity     INT NOT NULL,
  amount       INT NOT NULL,
  order_date   DATE NOT NULL,
  status       VARCHAR(10) NOT NULL DEFAULT '결제완료',
  updated_at   TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP ___③___   -- ③ 고칠 때마다 시각이 바뀌게
)"""
con = db()
cur = con.cursor()
cur.execute(DDL)
cur.execute("DELETE FROM orders")                         # 셀을 다시 실행할 때를 위해 비우기

rows = [(int(r.order_id), int(r.customer_id), r.category, int(r.price), int(r.quantity), int(r.amount), r.order_date.date())
        for r in orders.itertuples()]
cur.___④___("INSERT INTO orders (order_id, customer_id, category, price, quantity, amount, order_date) "   # ④ 여러 행을 한 번에
                "VALUES (%s, %s, %s, %s, %s, %s, %s)", rows)
cur.execute("SELECT count(*), sum(amount), min(order_date), max(order_date) FROM orders")
print(cur.fetchone())

> **확인 포인트** — `(1000, Decimal('291021500'), datetime.date(2026, 6, 1), datetime.date(2026, 7, 30))` — 1단계 pandas 의 합계와 같습니다.

In [ ]:
cur.execute("SELECT count(*), sum(amount) FROM orders")
assert cur.fetchone() == (1000, 291021500)

Debezium 은 binlog 를 읽는 전용 계정으로 붙습니다. 읽기 · 복제 권한만 주고 쓰기 권한은 주지 않습니다(6주차 6-4 2단계).

In [ ]:
cur.execute("CREATE USER IF NOT EXISTS 'debezium'@'%' IDENTIFIED BY 'dbz'")
cur.execute("GRANT SELECT, RELOAD, SHOW DATABASES, ___⑤___, REPLICATION CLIENT ON *.* TO 'debezium'@'%'")   # ⑤ binlog 를 읽는 복제 권한
cur.execute("SHOW VARIABLES LIKE 'binlog_format'")
print(cur.fetchone())

`binlog_format` 이 `ROW` 여야 행 단위 변경(무엇이 무엇으로 바뀌었나)이 기록됩니다. MySQL 8 의 기본값입니다.

일부러 같은 주문 번호를 한 번 더 넣어 보세요. 기본 키가 중복을 막습니다 — 6단계에서 반영기가 「같은 주문은 한 행」으로 맞추는 기준도 이 키입니다.

In [ ]:
try:
    cur.execute("INSERT INTO orders (order_id, customer_id, category, price, quantity, amount, order_date) "
                "VALUES (1, 1, '식품', 1000, 1, 1000, '2026-06-01')")
except pymysql.err.IntegrityError as e:
    print("넣기 실패:", e.args)

## 4단계 · Debezium 커넥터 등록 — 변경을 Kafka 로 (수집 · 스트리밍)

커넥터 설정은 6주차 6-4 3단계의 `register-mysql.json` 과 같고, 두 가지를 더했습니다.
`table.include.list` 로 `orders` 표만 잡고, 변환기의 `schemas.enable` 을 꺼서 이벤트마다 붙는 긴 스키마 설명을 뺍니다(6-5 의 Sink 설정과 같은 생각).

In [ ]:
cfg = {
    "connector.class": "io.debezium.connector.mysql.MySqlConnector",
    "tasks.max": "1",
    "database.hostname": "___⑥___",   # ⑥ Connect 컨테이너 기준 DB 호스트
    "database.port": "3306",
    "database.user": "debezium",
    "database.password": "dbz",
    "database.server.id": "184054",
    "topic.prefix": "shop",
    "database.include.list": "shop",
    "table.include.list": "shop.orders",
    "schema.history.internal.kafka.bootstrap.servers": "kafka:29092",
    "schema.history.internal.kafka.topic": "schema-changes.shop",
    "key.converter": "org.apache.kafka.connect.json.JsonConverter",
    "key.converter.schemas.enable": "false",
    "value.converter": "org.apache.kafka.connect.json.JsonConverter",
    "value.converter.schemas.enable": "false",
}
r = requests.___⑦___(f"{CONNECT}/connectors/shop-cdc/config", json=cfg, timeout=30)   # ⑦ 이름으로 등록 · 수정(있으면 덮어쓰기)
print(r.status_code)

def state():
    s = requests.get(f"{CONNECT}/connectors/shop-cdc/status", timeout=5).json()
    assert s["tasks"] and s["tasks"][0]["state"] == "RUNNING", s
    return s["connector"]["state"], s["tasks"][0]["state"]
print(wait(state, 120, "커넥터"))

`POST /connectors` 대신 `PUT /connectors/이름/config` 를 썼습니다. 처음이면 만들고, 이미 있으면 설정을 바꿉니다 — 셀을 다시 실행해도 「이미 있다(409)」 오류가 나지 않습니다.

> **확인 포인트** — `201` 과 `('RUNNING', 'RUNNING')` — 커넥터와 태스크가 모두 RUNNING 입니다(6주차 6-4 의 확인 포인트와 같음).

In [ ]:
assert r.status_code in (200, 201) and state() == ("RUNNING", "RUNNING")

커넥터가 처음 뜨면 표 전체를 한 번 읽는 **스냅숏**부터 합니다. 그 1,000건이 토픽 `shop.shop.orders`(접두사.DB.표)에 `op = r`(read) 이벤트로 들어갑니다.

일부러 커넥터 클래스 이름을 틀려 보세요. Connect 가 설정을 검사해 바로 거절합니다.

In [ ]:
try:
    bad = requests.put(f"{CONNECT}/connectors/bad-cdc/config", json={**cfg, "connector.class": "io.debezium.connector.mysql.MySql"}, timeout=30)
    bad.raise_for_status()
except requests.HTTPError as e:
    print("등록 실패:", bad.status_code, bad.json()["message"][:70])

## 5단계 · Iceberg 반영 — 스냅숏 1,000건을 레이크하우스로 (레이크 · Spark)

반영할 곳은 MinIO 위의 Iceberg 테이블입니다(18장 · 19장). 18장에서는 Spark 로 다뤘는데, 여기서는 같은 Iceberg 표를 파이썬 라이브러리 **PyIceberg** 로 씁니다 — JVM 없이 반영기 하나만 가볍게 돌리려는 것입니다.
카탈로그(표 이름 → 메타데이터 위치를 적은 전화번호부, 21장)는 SQLite 파일 하나로 둡니다.

In [ ]:
from minio import Minio
from pyiceberg.catalog.sql import SqlCatalog
from pyiceberg.expressions import In, EqualTo

mc = Minio("localhost:9000", access_key="minioadmin", secret_key="minioadmin", secure=False)
if not mc.bucket_exists("lakehouse"):
    mc.make_bucket("lakehouse")

cat = SqlCatalog("shop", uri="sqlite:///shop_catalog.db", warehouse="s3://lakehouse",
                 **{"s3.endpoint": "http://localhost:9000", "s3.access-key-id": "minioadmin",
                    "s3.secret-access-key": "minioadmin", "s3.region": "us-east-1"})
cat.create_namespace_if_not_exists("shop")

ARROW = pa.schema([("order_id", pa.int64()), ("customer_id", pa.int64()), ("category", pa.string()),
                   ("price", pa.int64()), ("quantity", pa.int64()), ("amount", pa.int64()),
                   ("order_date", pa.date32()), ("status", pa.string()), ("updated_at", pa.timestamp("us", tz="UTC"))])
if cat.table_exists("shop.orders"):
    cat.drop_table("shop.orders")
tbl = cat.create_table("shop.orders", schema=ARROW)
print(tbl.name(), len(tbl.schema().fields), "열")

이제 반영기입니다. Debezium 이벤트 하나는 `op`(c 생성 · u 수정 · d 삭제 · r 스냅숏)와 바뀌기 전 `before` · 바뀐 뒤 `after` 를 담고 있습니다(6주차 6-4).
반영기는 이벤트를 모아 **주문마다 마지막 이벤트만** 남긴 뒤, 살아 있는 주문은 upsert(있으면 고치고 없으면 넣기), 지워진 주문은 delete 합니다.

In [ ]:
from confluent_kafka import Consumer, KafkaException

EPOCH = date(1970, 1, 1)

def to_row(a):
    """Debezium after → Iceberg 한 행. DATE 는 1970-01-01 부터 센 날수, TIMESTAMP 는 ISO 글자로 옵니다."""
    return {"order_id": a["order_id"], "customer_id": a["customer_id"], "category": a["category"],
            "price": a["price"], "quantity": a["quantity"], "amount": a["amount"],
            "order_date": EPOCH + timedelta(days=a["order_date"]), "status": a["status"],
            "updated_at": datetime.fromisoformat(a["updated_at"].replace("Z", "+00:00"))}

def apply_cdc(consumer, tbl, idle_limit=10):
    ops, last, idle = Counter(), {}, 0
    while idle < idle_limit:                               # 10초 동안 새 이벤트가 없으면 이번 묶음은 끝
        m = consumer.poll(1.0)
        if m is None:
            idle += 1
            continue
        idle = 0
        if m.value() is None:                              # 삭제 뒤에 오는 빈 메시지(tombstone)
            ops["tombstone"] += 1
            continue
        ev = json.loads(m.value())
        key = (ev["after"] or ev["before"])["order_id"]
        last[key] = ev                                     # 같은 주문은 마지막 이벤트만
        ops[ev["___⑧___"]] += 1   # ⑧ 이벤트 종류가 든 필드
    ups = [to_row(e["after"]) for e in last.values() if e["op"] != "d"]
    dels = [k for k, e in last.items() if e["op"] == "d"]
    res = None
    if ups:
        res = tbl.___⑨___(pa.Table.from_pylist(ups, schema=ARROW), join_cols=["order_id"])   # ⑨ 있으면 고치고 없으면 넣기
    if dels:
        tbl.delete(delete_filter=___⑩___("order_id", dels))   # ⑩ 목록 안에 있는 것
    if ops:
        consumer.commit(asynchronous=False)                # 레이크에 쓴 뒤에 커밋 — 쓰다 죽으면 다시 읽습니다
    return ops, res, dels

consumer = Consumer({"bootstrap.servers": "localhost:9092", "group.id": "t3-lakehouse", "log_level": 3,
                     "auto.offset.reset": "earliest", "enable.auto.commit": False})
consumer.subscribe(["shop.shop.orders"])
ops, res, dels = apply_cdc(consumer, tbl)
print(dict(ops), "· 넣음", res.rows_inserted, "· 고침", res.rows_updated)

> **확인 포인트** — `{'r': 1000} · 넣음 1000 · 고침 0` — 스냅숏 이벤트 1,000건이 모두 새 행으로 들어갔습니다.

In [ ]:
snap0 = tbl.current_snapshot().snapshot_id                 # 6단계에서 시간 여행할 때 씁니다
lake = tbl.scan().to_arrow()
assert dict(ops) == {"r": 1000} and res.rows_inserted == 1000 and lake.num_rows == 1000
assert duckdb.sql("SELECT sum(amount) FROM lake").fetchall()[0][0] == 291021500

DuckDB 는 Iceberg 에서 읽어 온 Arrow 표 이름(`lake`)을 그대로 SQL 의 테이블로 씁니다(14장). 금액 합계가 MySQL 과 같습니다.

일부러 표에 없는 열(`coupon`)이 든 행을 넣어 보세요. Iceberg 는 스키마에 없는 열을 거절합니다 — 열을 늘리려면 먼저 스키마를 바꿉니다(18장 스키마 진화).
반대로 **열이 빠진** 행은 거절하지 않고 빈 값(null)으로 넣어 버립니다(모든 열이 optional 이라서). 반영기가 늘 `ARROW` 스키마로 표를 만들어 넣는 이유입니다.

In [ ]:
try:
    tbl.append(pa.table({"order_id": [9999], "status": ["결제완료"], "coupon": ["WELCOME"]}))
except ValueError as e:
    print("쓰기 실패:", str(e).splitlines()[0][:80])
assert tbl.scan().to_arrow().num_rows == 1000

## 6단계 · 주문 상태 바꾸기 — 변경이 레이크하우스까지 오는지 (레이크 · Spark)

이제 쇼핑몰이 돌아가는 모습을 흉내 냅니다. 아래 변경은 모두 **연습용으로 만든 갱신**입니다.

| 변경 | SQL | 이벤트 |
|---|---|---|
| 1~300번 출고 | `UPDATE … SET status='배송중' WHERE order_id <= 300` | u 300건 |
| 1~100번 도착 | `UPDATE … SET status='배송완료' WHERE order_id <= 100` | u 100건 |
| 7 · 42번 환불 요청 | `UPDATE … SET status='환불요청' WHERE order_id IN (7, 42)` | u 2건 |
| 새 주문 1001번 | `INSERT …` | c 1건 |
| 시험 주문 1002번을 넣었다 지움 | `INSERT …` → `DELETE …` | c 1건 · d 1건 |

In [ ]:
cur.execute("UPDATE orders SET status='배송중' WHERE order_id <= 300")
cur.execute("UPDATE orders SET status='배송완료' WHERE order_id <= 100")
cur.execute("UPDATE orders SET status='환불요청' WHERE order_id IN (7, 42)")
cur.execute("INSERT INTO orders (order_id, customer_id, category, price, quantity, amount, order_date) "
            "VALUES (1001, 12, '도서', 18000, 2, 36000, '2026-10-28')")
cur.execute("INSERT INTO orders (order_id, customer_id, category, price, quantity, amount, order_date) "
            "VALUES (1002, 1, '식품', 1000, 1, 1000, '2026-10-28')")
cur.execute("DELETE FROM orders WHERE order_id = 1002")
cur.execute("SELECT status, count(*) FROM orders GROUP BY status ORDER BY 2 DESC")
src = dict(cur.fetchall())
print("MySQL:", src)

MySQL 에서는 이미 바뀌었습니다. 반영기를 한 번 더 돌려 Kafka 에 쌓인 변경을 레이크하우스로 가져옵니다. 5단계에서 커밋한 자리 다음부터 읽으므로 스냅숏 1,000건은 다시 오지 않습니다.

In [ ]:
ops, res, dels = apply_cdc(consumer, tbl)
print(dict(ops), "· 넣음", res.rows_inserted, "· 고침", res.rows_updated, "· 지움", dels)

lake = tbl.scan().to_arrow()
now = duckdb.sql("SELECT status, count(*) AS n FROM lake GROUP BY status ORDER BY n DESC").df()
print("Iceberg:", dict(zip(now["status"], now["n"])))

같은 주문이 두 번 바뀌어도(1~100번은 배송중 → 배송완료) 마지막 상태 하나만 반영됩니다. 그래서 이벤트는 402건인데 고친 행은 300건입니다.
1002번은 넣었다 지웠으므로 마지막 이벤트가 `d` — 레이크하우스에는 처음부터 없던 것과 같습니다. 그래서 지울 행이 없다는 경고(`Delete operation did not match any records`)가 한 줄 나오는데, 정상입니다.

> **확인 포인트** — 이벤트 `{'u': 402, 'c': 2, 'd': 1, 'tombstone': 1}` · 넣음 1 · 고침 300 · 지움 `[1002]`.
> 두 쪽 상태가 같습니다 — `결제완료 701 · 배송중 200 · 배송완료 98 · 환불요청 2`, 모두 1,001행.

In [ ]:
assert dict(ops) == {"u": 402, "c": 2, "d": 1, "tombstone": 1}
assert res.rows_inserted == 1 and res.rows_updated == 300 and dels == [1002]
assert dict(zip(now["status"], now["n"])) == src == {"결제완료": 701, "배송중": 200, "배송완료": 98, "환불요청": 2}
assert lake.num_rows == 1001

이 확인이 이 프로젝트에서 가장 중요합니다 — **원본 DB 와 레이크하우스가 같은 상태인가**(정합성 대조). 10단계의 Airflow DAG 도 매번 이것을 검사합니다.

Iceberg 는 쓸 때마다 스냅숏을 남깁니다(18장 타임 트래블). 5단계의 스냅숏으로 돌아가면 배송이 시작되기 전 모습이 그대로 있습니다.

In [ ]:
before = tbl.scan(snapshot_id=___⑪___).to_arrow()   # ⑪ 5단계에서 적어 둔 스냅숏
print("스냅숏", len(tbl.snapshots()), "개 · 처음 모습:", before.num_rows, "행", set(before.column("status").to_pylist()))
assert before.num_rows == 1000 and set(before.column("status").to_pylist()) == {"결제완료"}

일부러 tombstone 을 그대로 읽어 보세요. 삭제 이벤트 뒤의 빈 메시지(값이 `None`)를 거르지 않으면 반영기가 멈춥니다 — `apply_cdc` 의 `if m.value() is None` 이 그래서 있습니다.

In [ ]:
try:
    json.loads(None)
except TypeError as e:
    print("읽기 실패:", e)

## 7단계 · 규정 문서 — 가상 쇼핑몰 FAQ 를 Qdrant 에 (RAG · 서비스)

상담봇이 기댈 규정 문서를 씁니다. 아래 「데이터몰」은 **이 과정의 연습용 가상 쇼핑몰**이고, 규정도 실습을 위해 지어낸 것입니다.

In [ ]:
%%writefile shop_policy.md
# 데이터몰 고객 안내 (연습용 가상 쇼핑몰)

## 배송 안내
결제완료 상태의 주문은 영업일 기준 1~2일 안에 출고합니다. 출고되면 주문 상태가 배송중으로 바뀝니다.
배송중 주문은 보통 출고 다음 날부터 1~2일 안에 도착합니다. 도서 · 산간 지역은 1~2일 더 걸릴 수 있습니다.
도착하면 상태가 배송완료로 바뀝니다. 주문 번호로 지금 상태를 확인할 수 있습니다.

## 교환 · 환불 규정
상품을 받은 날부터 7일 안에 교환이나 환불을 신청할 수 있습니다.
환불은 반품된 상품을 확인한 뒤 3영업일 안에 결제한 수단으로 돌려드립니다. 신청하면 주문 상태가 환불요청으로 바뀝니다.
단순 변심으로 반품하면 왕복 배송비 6,000원을 뺀 금액을 돌려드립니다. 상품 불량이나 잘못 보낸 경우에는 배송비를 받지 않습니다.
식품은 상하기 쉬워 단순 변심 환불을 받지 않습니다. 포장을 뜯은 도서도 교환 · 환불이 어렵습니다.

## 주문 취소
결제완료 상태에서는 마이페이지에서 바로 취소할 수 있고, 결제는 즉시 취소됩니다.
배송중으로 바뀐 뒤에는 취소할 수 없습니다. 상품을 받은 뒤 교환 · 환불 규정에 따라 반품해 주세요.

## 배송비
한 주문이 30,000원 이상이면 배송비가 없습니다. 30,000원 미만이면 3,000원입니다.
제주와 도서 · 산간 지역은 3,000원이 더 붙습니다.

## 상품 불량 · 파손
상품이 파손되었거나 불량이면 받은 날부터 30일 안에 사진과 함께 알려 주세요. 새 상품으로 바꾸거나 전액 환불합니다.
가전은 제조사 보증 기간 동안 제조사 서비스 센터에서 무상 수리를 받을 수 있습니다.

## 적립금
결제 금액의 1%를 적립금으로 드립니다. 적립금은 배송완료 뒤 7일이 지나면 쓸 수 있습니다.
환불하면 그 주문으로 받은 적립금은 빠집니다.

## 상담봇 이용 안내
상담봇은 로그인한 고객 본인의 주문만 조회합니다. 다른 사람의 주문 번호를 넣으면 조회하지 않습니다.
상담봇이 답하기 어려운 내용은 고객센터 상담원에게 연결해 드립니다.

문서를 `## 제목` 단위로 자릅니다. 규정 문서는 제목 하나가 주제 하나라, 제목 단위로 자르면 한 조각 안에 다른 규정이 섞이지 않습니다(7주차 청킹 — 「의미 단위로 자르기」).

In [ ]:
text = open("shop_policy.md", encoding="utf-8").read()
chunks = []
for part in re.split(r"\n(?=___⑫___ )", text)[1:]:   # ⑫ 둘째 단계 제목 표시
    title, body = part.split("\n", 1)
    chunks.append({"section": title.lstrip("# ").strip(), "text": title.lstrip("# ").strip() + "\n" + body.strip()})
print(len(chunks), "조각")
[c["section"] for c in chunks]

> **확인 포인트** — 7조각 — `배송 안내` · `교환 · 환불 규정` · `주문 취소` · `배송비` · `상품 불량 · 파손` · `적립금` · `상담봇 이용 안내`.

In [ ]:
assert len(chunks) == 7 and chunks[1]["section"] == "교환 · 환불 규정"

> **Aebon's Tip** — 문서가 길어 한 제목 아래 글이 수천 자가 되면, 7주차의 `RecursiveCharacterTextSplitter`(chunk_size · chunk_overlap)로 한 번 더 자르세요. 그때도 조각마다 `section` 을 메타데이터로 남겨 두면 답에 「어느 규정인지」를 붙일 수 있습니다.

In [ ]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

emb = SentenceTransformer("jhgan/ko-sroberta-multitask")
qd = QdrantClient(url="http://localhost:6333")
if qd.collection_exists("t3_policy"):
    qd.delete_collection("t3_policy")
qd.create_collection("t3_policy", vectors_config=VectorParams(size=768, distance=Distance.COSINE))
vecs = emb.encode([c["text"] for c in chunks], normalize_embeddings=True)
qd.upsert("t3_policy", points=[PointStruct(id=i, vector=v.tolist(), payload=c) for i, (c, v) in enumerate(zip(chunks, vecs))])

for q in ["환불 규정은?", "내 주문 언제 와?", "주문 취소할 수 있어?"]:
    top = qd.query_points("t3_policy", query=emb.encode(q, normalize_embeddings=True).tolist(), limit=1).points[0]
    print(f"{q:12} → {top.payload['section']} ({top.score:.2f})")

> **확인 포인트** — `환불 규정은? → 교환 · 환불 규정` · `내 주문 언제 와? → 배송 안내` · `주문 취소할 수 있어? → 주문 취소`.

In [ ]:
def top1(q):
    return qd.query_points("t3_policy", query=emb.encode(q, normalize_embeddings=True).tolist(), limit=1).points[0].payload["section"]
assert qd.count("t3_policy").count == 7
assert top1("환불 규정은?") == "교환 · 환불 규정" and top1("내 주문 언제 와?") == "배송 안내" and top1("주문 취소할 수 있어?") == "주문 취소"

> **Aebon's Tip** — 「배송비 얼마야?」를 넣어 보세요. 1위가 `배송비` 가 아니라 `배송 안내`(0.57 대 0.56)로 나옵니다. 「배송」이라는 낱말을 함께 가진 두 조각을 임베딩만으로는 잘 가르지 못하는 것입니다.
> 7주차의 BM25(낱말 일치 점수)를 섞는 하이브리드 검색으로 바꾸면 「배송비」라는 낱말이 그대로 든 조각이 올라옵니다 — RAG 담당이 해 볼 개선 과제입니다.

일부러 없는 컬렉션에서 찾아 보세요.

In [ ]:
try:
    qd.query_points("t3_faq", query=[0.0] * 768, limit=1)
except Exception as e:
    print("검색 실패:", type(e).__name__, re.search(r'"error":"([^"]+)', str(e)).group(1)[:60] if '"error"' in str(e) else str(e)[:60])

## 8단계 · 상담봇 — 주문 조회(SQL)와 규정 검색(RAG) 엮기 (RAG · 서비스)

상담봇은 질문마다 길을 고릅니다(라우팅).

- 질문에 **주문 번호**가 있으면 → 레이크하우스에서 그 주문을 SQL 로 찾고, 상태에 맞는 규정(배송 · 환불)도 함께 붙입니다.
- 번호가 없으면 → 규정 검색(RAG)만 합니다.

주문 조회에는 **고객 번호 조건**을 꼭 붙입니다. 「내 주문」만 보여 줘야 하기 때문입니다(규정의 「상담봇 이용 안내」). SQL 값은 `?` 자리표시로 넘깁니다 — 질문 글을 SQL 문자열에 이어 붙이면 SQL 인젝션이 됩니다(7장).

봇은 Streamlit 화면(9단계)과 노트북이 함께 쓰도록 `bot.py` 파일로 만듭니다.

In [ ]:
%%writefile bot.py
"""T3 쇼핑몰 상담봇 — 주문 조회(SQL, Iceberg) + 규정 검색(RAG, Qdrant)."""
import os, re
import duckdb

_cache = {}

def load():
    if not _cache:
        from pyiceberg.catalog.sql import SqlCatalog
        from qdrant_client import QdrantClient
        from sentence_transformers import SentenceTransformer
        cat = SqlCatalog("shop", uri="sqlite:///shop_catalog.db", warehouse="s3://lakehouse",
                         **{"s3.endpoint": os.environ.get("T3_S3", "http://localhost:9000"), "s3.access-key-id": "minioadmin",
                            "s3.secret-access-key": "minioadmin", "s3.region": "us-east-1"})
        _cache.update(tbl=cat.load_table("shop.orders"), emb=SentenceTransformer("jhgan/ko-sroberta-multitask"),
                      qd=QdrantClient(url=os.environ.get("T3_QDRANT", "http://localhost:6333")))
    return _cache

def lookup(order_id, customer_id):
    """주문 조회 — 본인 주문만. 레이크하우스의 최신 스냅숏을 DuckDB SQL 로 읽습니다."""
    lake = load()["tbl"].refresh().scan().to_arrow()
    df = duckdb.execute("SELECT order_id, category, amount, order_date, status, updated_at FROM lake "
                        "WHERE order_id = ? AND customer_id = ___⑬___", [order_id, customer_id]).df()   # ⑬ 값은 자리표시로
    return df.iloc[0].to_dict() if len(df) else None

def search(q, k=2):
    c = load()
    hits = c["qd"].query_points("t3_policy", query=c["emb"].encode(q, normalize_embeddings=True).tolist(), limit=k).points
    return [h.payload | {"score": round(h.score, 2)} for h in hits]

HINT = {"결제완료": "결제완료", "배송중": "배송중", "배송완료": "7일 안에", "환불요청": "3영업일"}

def pick(doc_text, status):
    """규정 조각에서 주문 상태에 맞는 문장 하나를 고릅니다."""
    sents = re.split(r"(?<=다\.)\s+", doc_text.split("\n", 1)[1])
    return next((x for x in sents if x.startswith(HINT[status])), next((x for x in sents if HINT[status] in x), sents[0]))

STATUS_Q = {"결제완료": "결제완료 주문은 언제 출고되나요?", "배송중": "배송중 주문은 언제 도착하나요?",
            "배송완료": "교환이나 환불은 어떻게 하나요?", "환불요청": "환불은 언제 돌려받나요?"}

def answer(question, customer_id):
    m = re.search(r"___⑭___", question)   # ⑭ 숫자 덩어리 + 「번」
    if m:
        order = lookup(int(m.group(1)), customer_id)
        if order is None:
            return {"route": "order", "order": None, "docs": search("상담봇은 본인 주문만 조회", 1),
                    "text": f"{m.group(1)}번은 고객님의 주문이 아니거나 없는 주문입니다. 본인 주문 번호를 확인해 주세요."}
        docs = search(STATUS_Q[order["status"]] + " " + question)
        return {"route": "order", "order": order, "docs": docs,
                "text": f"{order['order_id']}번 주문({order['category']}, {order['amount']:,}원)은 지금 '{order['status']}' 상태입니다. "
                        + pick(docs[0]["text"], order["status"])}
    docs = search(question)
    return {"route": "rag", "order": None, "docs": docs, "text": docs[0]["text"].split("\n", 1)[1]}

def llm(result, question):
    """OPENAI_API_KEY 가 있을 때만 — 근거(주문 + 규정)만으로 문장을 다듬습니다. 없으면 None."""
    key = os.environ.get("OPENAI_API_KEY")
    if not key:
        return None
    from openai import OpenAI
    ctx = f"[주문] {result['order']}\n" + "\n".join(f"[규정: {d['section']}] {d['text']}" for d in result["docs"])
    r = OpenAI(api_key=key).chat.completions.create(model="gpt-4o-mini", temperature=0, messages=[
        {"role": "system", "content": "데이터몰 상담원입니다. 근거에 있는 내용만으로 짧고 친절하게 답하고, 근거가 없으면 상담원 연결을 안내하세요."},
        {"role": "user", "content": ctx + "\n\n질문: " + question}])
    return r.choices[0].message.content

고객 12번이 묻는다고 해 봅니다. 12번의 주문은 150번(식품)과 6단계에서 새로 들어온 1001번입니다.

In [ ]:
import importlib, bot
importlib.reload(bot)

for q in ["150번 주문 언제 와?", "1001번 주문 언제 와?", "환불 규정은?"]:
    r = bot.answer(q, customer_id=12)
    print(f"Q. {q}  [{r['route']}] 근거: {[d['section'] for d in r['docs']]}")
    print("A.", r["text"], "\n")

> **확인 포인트** — 150번은 `배송중`(6단계에서 1~300번을 출고), 1001번은 `결제완료` 이고 둘 다 근거 1위가 `배송 안내` 입니다. 「환불 규정은?」은 주문 조회 없이(`rag`) `교환 · 환불 규정` 으로 답합니다.

In [ ]:
r150, r1001, rref = bot.answer("150번 주문 언제 와?", 12), bot.answer("1001번 주문 언제 와?", 12), bot.answer("환불 규정은?", 12)
assert r150["order"]["status"] == "배송중" and r150["docs"][0]["section"] == "배송 안내"
assert r1001["order"]["status"] == "결제완료" and r1001["docs"][0]["section"] == "배송 안내"
assert rref["route"] == "rag" and rref["order"] is None and rref["docs"][0]["section"] == "교환 · 환불 규정"

일부러 남의 주문을 물어보세요. 17번은 고객 103번의 주문입니다. 고객 번호 조건 때문에 조회되지 않습니다.

In [ ]:
r17 = bot.answer("17번 주문 언제 와?", customer_id=12)
print(r17["text"])
assert r17["order"] is None and bot.lookup(17, 103)["status"] == "배송완료"

답 문장을 LLM 으로 다듬는 것은 키가 있을 때만입니다(7주차 `gpt-4o-mini`). 없으면 위의 규칙형 답으로 충분히 동작합니다.

In [ ]:
print(bot.llm(r150, "150번 주문 언제 와?") or "OPENAI_API_KEY 가 없어 규칙형 답을 씁니다: " + r150["text"])

## 9단계 · Streamlit 화면 — 상담 창 (RAG · 서비스)

8주차의 채팅 화면(`st.chat_input` · `st.chat_message`)에 `bot.answer` 를 붙입니다. 무거운 준비(모델 · 테이블 읽기)는 `st.cache_resource` 로 한 번만 합니다.

In [ ]:
%%writefile app.py
import streamlit as st
import bot

st.set_page_config(page_title="데이터몰 상담봇")
st.title("데이터몰 상담봇")
st.caption("연습용 가상 쇼핑몰입니다. 주문 번호와 함께 물어보세요. 예: 150번 주문 언제 와?")

@st.___⑮___   # ⑮ 모델 · 연결을 한 번만 만들기
def ready():
    return bot.load()

ready()
customer = st.sidebar.number_input("고객 번호(로그인한 사용자)", min_value=1, value=12)

if "history" not in st.session_state:
    st.session_state.history = []
for role, text in st.session_state.history:
    st.chat_message(role).write(text)

q = st.chat_input("무엇을 도와드릴까요?")
if q:
    st.chat_message("user").write(q)
    r = bot.answer(q, int(customer))
    text = bot.llm(r, q) or r["text"]
    with st.chat_message("assistant"):
        st.write(text)
        if r["order"]:
            st.json({k: str(v) for k, v in r["order"].items()})
        st.caption("근거: " + ", ".join(d["section"] for d in r["docs"]))
    st.session_state.history += [("user", q), ("assistant", text)]

내 컴퓨터에서 화면을 열 때는 터미널에서 아래처럼 실행하고 브라우저로 http://localhost:8501 에 들어갑니다(노트북의 작업 폴더에서).

```bash
streamlit run app.py
```

노트북 안에서는 Streamlit 의 `AppTest` 로 화면을 띄우지 않고 실행해 봅니다. 질문을 넣고 나온 답 글을 읽어 확인합니다.

In [ ]:
from streamlit.testing.v1 import AppTest

at = AppTest.from_file(os.path.abspath("app.py"), default_timeout=180).run()   # 상대 경로는 호출한 파일 기준이라 절대 경로로
at.chat_input[0].set_value("150번 주문 언제 와?").run()
said = [m.markdown[0].value for m in at.chat_message if m.markdown]
print("화면 제목:", at.title[0].value)
print("봇 답:", said[-1][:80])
print("근거 표시:", at.caption[-1].value)

> **확인 포인트** — 제목 `데이터몰 상담봇`, 봇 답에 `150번 주문` · `배송중` 이 들어 있고, 근거 표시가 `근거: 배송 안내, …` 로 시작합니다. 예외는 0건입니다.

In [ ]:
assert not at.exception and at.title[0].value == "데이터몰 상담봇"
assert "배송중" in said[-1] and at.caption[-1].value.startswith("근거: 배송 안내")

## 10단계 · Airflow DAG — 반영과 정합성 대조를 주기적으로 (오케스트레이션)

6단계에서 손으로 돌린 반영기를 Airflow 가 10분마다 돌리게 하고, 매번 **MySQL 과 Iceberg 의 행 수 · 상태별 개수가 같은지** 대조합니다. 다르면 태스크가 실패하고, 재시도 뒤에도 다르면 사람이 볼 수 있게 멈춥니다(8주차).

반영기 함수는 노트북에서 쓴 것을 그대로 파일로 옮깁니다(`inspect.getsource`).

In [ ]:
import inspect
os.makedirs("dags", exist_ok=True)
open("dags/t3_sink.py", "w", encoding="utf-8").write(
    "import json\nfrom collections import Counter\nfrom datetime import date, datetime, timedelta\nimport pyarrow as pa\n"
    "from pyiceberg.expressions import In\n\nEPOCH = date(1970, 1, 1)\n"
    "ARROW = pa.schema([('order_id', pa.int64()), ('customer_id', pa.int64()), ('category', pa.string()), ('price', pa.int64()),\n"
    "                   ('quantity', pa.int64()), ('amount', pa.int64()), ('order_date', pa.date32()), ('status', pa.string()),\n"
    "                   ('updated_at', pa.timestamp('us', tz='UTC'))])\n\n\n"
    + inspect.getsource(to_row) + "\n\n" + inspect.getsource(apply_cdc))
print(sorted(os.listdir("dags")))

In [ ]:
%%writefile dags/t3_cdc_lakehouse.py
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator

with DAG(
    dag_id="t3_cdc_lakehouse",
    start_date=datetime(2026, 11, 1),
    schedule="___⑯___",   # ⑯ 10분마다(크론)
    catchup=False,
    default_args={"owner": "t3", "retries": 2, "retry_delay": timedelta(minutes=2)},
    tags=["T3", "cdc"],
) as dag:
    def _apply(**_):
        from confluent_kafka import Consumer
        from pyiceberg.catalog.sql import SqlCatalog
        import t3_sink
        tbl = SqlCatalog("shop", uri="sqlite:////opt/t3/shop_catalog.db", warehouse="s3://lakehouse",
                         **{"s3.endpoint": "http://minio:9000", "s3.access-key-id": "minioadmin",
                            "s3.secret-access-key": "minioadmin", "s3.region": "us-east-1"}).load_table("shop.orders")
        c = Consumer({"bootstrap.servers": "kafka:29092", "group.id": "t3-lakehouse",
                      "auto.offset.reset": "earliest", "enable.auto.commit": False})
        c.subscribe(["shop.shop.orders"])
        ops, _, _ = t3_sink.apply_cdc(c, tbl)
        c.close()
        return dict(ops)

    def _reconcile(**_):
        import pymysql, duckdb
        from pyiceberg.catalog.sql import SqlCatalog
        con = pymysql.connect(host="mysql", port=3306, user="root", password="rootpw", database="shop")
        cur = con.cursor()
        cur.execute("SELECT status, count(*) FROM orders GROUP BY status")
        src = dict(cur.fetchall())
        lake = SqlCatalog("shop", uri="sqlite:////opt/t3/shop_catalog.db", warehouse="s3://lakehouse",
                          **{"s3.endpoint": "http://minio:9000", "s3.access-key-id": "minioadmin",
                             "s3.secret-access-key": "minioadmin", "s3.region": "us-east-1"}).load_table("shop.orders").scan().to_arrow()
        dst = dict(duckdb.sql("SELECT status, count(*) FROM lake GROUP BY status").fetchall())
        if src != dst:
            raise ValueError(f"정합성 불일치 MySQL={src} Iceberg={dst}")   # 실패 → 재시도 2번 → 멈춤
        return src

    apply = PythonOperator(task_id="apply_cdc", python_callable=_apply)
    reconcile = PythonOperator(task_id="reconcile", python_callable=_reconcile)
    apply >> reconcile

노트북 venv 에는 Airflow 가 없으므로 2단계 compose 의 `airflow` 서비스(8주차와 같은 2.9.3)를 잠깐 띄워 DAG 가 읽히는지 검사합니다. 처음에는 이미지(약 2GB)를 받느라 몇 분 걸릴 수 있습니다.

In [ ]:
out = subprocess.run(
    ["docker", "compose", "-p", "t3", "--profile", "airflow", "run", "--rm", "airflow", "bash", "-c",
     "airflow db migrate > /dev/null 2>&1 && airflow dags list-import-errors && airflow ___⑰___ t3_cdc_lakehouse"],   # ⑰ DAG 의 태스크 목록
    capture_output=True, text=True)
print(out.stdout[-400:])

> **확인 포인트** — 읽기 오류는 `No data found`, 태스크는 `apply_cdc` · `reconcile` 둘입니다.

In [ ]:
assert out.returncode == 0 and "No data found" in out.stdout
assert [t for t in out.stdout.split() if t in ("apply_cdc", "reconcile")] == ["apply_cdc", "reconcile"]

정합성 대조가 실제로 어긋남을 잡는지, 일부러 MySQL 만 바꾸고 반영기를 돌리지 않은 채 대조해 보세요.

In [ ]:
def reconcile():
    cur.execute("SELECT status, count(*) FROM orders GROUP BY status")
    src = dict(cur.fetchall())
    dst = dict(duckdb.sql("SELECT status, count(*) FROM lake GROUP BY status").fetchall())
    if src != dst:
        raise ValueError(f"정합성 불일치 MySQL={src} Iceberg={dst}")
    return src

cur.execute("UPDATE orders SET status='배송완료' WHERE order_id = 150")
try:
    reconcile()
except ValueError as e:
    print("대조 실패:", str(e)[:90])

apply_cdc(consumer, tbl)
lake = tbl.scan().to_arrow()
print("반영 뒤:", reconcile())

> **확인 포인트** — 처음에는 `정합성 불일치` 로 실패하고, 반영기를 한 번 돌린 뒤에는 `배송중 199 · 배송완료 99` 로 두 쪽이 같아집니다.

In [ ]:
assert reconcile()["배송완료"] == 99 and bot.lookup(150, 12)["status"] == "배송완료"

> **Aebon's Tip** — 운영에서는 반영기를 10분 주기 배치가 아니라 **늘 떠 있는 소비자**(컨테이너 하나)로 두고, Airflow 는 정합성 대조와 Iceberg 정리(오래된 스냅숏 지우기 · 작은 파일 합치기)만 맡기는 구성이 흔합니다. 팀에서 둘 중 하나를 골라 이유를 발표 자료에 적어 보세요.

## 11단계 · 정리와 더 해 보기 (오케스트레이션)

커넥터를 지우고, 소비자를 닫고, 컨테이너와 볼륨을 지웁니다.

In [ ]:
requests.delete(f"{CONNECT}/connectors/shop-cdc", timeout=30)
consumer.close()
con.close()
!docker compose --progress quiet -p t3 --profile airflow ___⑱___   # ⑱ 컨테이너와 볼륨까지 지우기
left = subprocess.run(["docker", "ps", "-a", "-q", "--filter", "label=com.docker.compose.project=t3"], capture_output=True, text=True).stdout.split()
print("남은 컨테이너:", len(left))
assert left == []

> **확인 포인트** — `남은 컨테이너: 0`.

더 해 보기 — 역할별로 하나씩 골라 키우세요.

- 수집 · 스트리밍: 1권 7장의 정규화된 쇼핑몰(customers · products · orders · order_items)을 그대로 MySQL 에 올리고, `table.include.list` 에 네 표를 모두 넣어 보세요. 토픽이 표마다 하나씩 생깁니다.
- 레이크 · Spark: 18장의 Spark + Iceberg 로 같은 테이블(`shop.orders`)을 읽어 일별 · 상태별 매출을 집계하세요. 카탈로그를 REST 카탈로그(21장)로 바꾸면 Spark 와 PyIceberg 가 같은 표를 함께 봅니다.
- RAG · 서비스: 「환불하면 적립금은?」처럼 규정 두 개가 함께 필요한 질문에서 근거를 2개 붙여 답하게 해 보세요. 질문 의도(주문 · 규정 · 상담원 연결)를 나누는 규칙을 늘리는 것도 좋습니다.
- 오케스트레이션: DAG 의 `reconcile` 이 실패했을 때 재시도 2번 뒤 멈추는지 Airflow 화면에서 확인하고, 실패 알림(메일 · 슬랙)을 붙여 보세요.

## 정답

① `3307:3306` — 내 노트북 3307 → 컨테이너 3306
② `kafka:29092` — 컨테이너 안에서 붙는 Kafka 주소
③ `ON UPDATE CURRENT_TIMESTAMP` — 고칠 때마다 시각이 바뀌게
④ `executemany` — 여러 행을 한 번에
⑤ `REPLICATION SLAVE` — binlog 를 읽는 복제 권한
⑥ `mysql` — Connect 컨테이너 기준 DB 호스트
⑦ `put` — 이름으로 등록 · 수정(있으면 덮어쓰기)
⑧ `op` — 이벤트 종류가 든 필드
⑨ `upsert` — 있으면 고치고 없으면 넣기
⑩ `In` — 목록 안에 있는 것
⑪ `snap0` — 5단계에서 적어 둔 스냅숏
⑫ `##` — 둘째 단계 제목 표시
⑬ `?` — 값은 자리표시로
⑭ `(\d+)\s*번` — 숫자 덩어리 + 「번」
⑮ `cache_resource` — 모델 · 연결을 한 번만 만들기
⑯ `*/10 * * * *` — 10분마다(크론)
⑰ `tasks list` — DAG 의 태스크 목록
⑱ `down -v` — 컨테이너와 볼륨까지 지우기